# 00 — Load Data
Downloads four Kaggle pharmacy/pharma datasets to `../data/` using `opendatasets`.
You will be prompted once for your **Kaggle username** and **API key**  
(get the key from https://www.kaggle.com/settings → *API* → *Create New Token*).

In [7]:
%pip install -q kaggle pandas python-dotenv openpyxl json5

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Load credentials from ../.env into the environment BEFORE importing kaggle.
# The kaggle client authenticates on import using KAGGLE_USERNAME / KAGGLE_KEY.
load_dotenv(Path("../.env"))

assert os.getenv("KAGGLE_USERNAME"), "KAGGLE_USERNAME missing — set it in .env"
assert os.getenv("KAGGLE_KEY"),      "KAGGLE_KEY missing — set it in .env"

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# (owner/dataset-slug, target subfolder)
DATASETS = [
    "milanzdravkovic/pharma-sales-data",
    "pritipoddar/inventory-data-for-pharmacy-website-in-json-format",
    "mohammedashraf000/pharmaceutical-supply-chain-optimization",
    "hossam82/pharmacy-products-dataset",
]

print(f"Authenticated as: {os.getenv('KAGGLE_USERNAME')}")
print(f"Data directory:   {DATA_DIR.resolve()}")

Authenticated as: jetanin
Data directory:   C:\Work\LogisticsInnovationHackathon2026\MedCast_Secure\data


In [9]:
def download(slug: str, base_dir: Path) -> Path:
    """Download + unzip a Kaggle dataset into base_dir/<name>/. Skips if present."""
    name = slug.split("/")[1]
    dest = base_dir / name
    if dest.exists() and any(dest.iterdir()):
        print(f"[skip] {slug} — already in {dest}")
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    print(f"[download] {slug}")
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    return dest


for slug in DATASETS:
    download(slug, DATA_DIR)

print("\nAll datasets downloaded to:", DATA_DIR.resolve())

[download] milanzdravkovic/pharma-sales-data
Dataset URL: https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data


100%|██████████| 352k/352k [00:00<00:00, 381kB/s]



[download] pritipoddar/inventory-data-for-pharmacy-website-in-json-format
Dataset URL: https://www.kaggle.com/datasets/pritipoddar/inventory-data-for-pharmacy-website-in-json-format


100%|██████████| 14.4k/14.4k [00:00<00:00, 4.27MB/s]


[download] mohammedashraf000/pharmaceutical-supply-chain-optimization
Dataset URL: https://www.kaggle.com/datasets/mohammedashraf000/pharmaceutical-supply-chain-optimization


100%|██████████| 2.32M/2.32M [00:01<00:00, 1.62MB/s]



[download] hossam82/pharmacy-products-dataset
Dataset URL: https://www.kaggle.com/datasets/hossam82/pharmacy-products-dataset


100%|██████████| 170k/170k [00:00<00:00, 234kB/s]



All datasets downloaded to: C:\Work\LogisticsInnovationHackathon2026\MedCast_Secure\data


## Convert all files to CSV
Converts every non-CSV file under `../data/` (`.xlsx`, `.json`, `.js`) into a `.csv`
sitting next to the original.

In [10]:
import re
import json5


def js_to_records(text: str):
    """Extract the array literal from a JS file (e.g. `const x = [ ... ];`) and parse it."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("No array literal found in JS file")
    return json5.loads(text[start : end + 1])


def to_dataframe(fp: Path) -> pd.DataFrame:
    suffix = fp.suffix.lower()
    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(fp)
    if suffix == ".json":
        with open(fp, encoding="utf-8") as f:
            data = json.load(f)
        return pd.json_normalize(data)
    if suffix == ".js":
        return pd.json_normalize(js_to_records(fp.read_text(encoding="utf-8")))
    raise ValueError(f"Unsupported file type: {fp.suffix}")


converted = []
for fp in sorted(DATA_DIR.rglob("*")):
    if not fp.is_file() or fp.suffix.lower() not in (".xlsx", ".xls", ".json", ".js"):
        continue
    out = fp.with_suffix(".csv")
    try:
        df = to_dataframe(fp)
        df.to_csv(out, index=False)
        converted.append(out)
        print(f"[ok] {fp.relative_to(DATA_DIR)} -> {out.name}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")
    except Exception as e:
        print(f"[fail] {fp.relative_to(DATA_DIR)}: {e}")

print(f"\nConverted {len(converted)} file(s) to CSV.")

[ok] inventory-data-for-pharmacy-website-in-json-format\data.js -> data.csv  (93 rows x 12 cols)
[ok] pharmaceutical-supply-chain-optimization\Pharmaceutical Supply Chain Optimization.xlsx -> Pharmaceutical Supply Chain Optimization.csv  (100,000 rows x 4 cols)

Converted 2 file(s) to CSV.


In [11]:
def load_file(fp: Path):
    if fp.suffix.lower() == ".csv":
        return pd.read_csv(fp)
    if fp.suffix.lower() == ".json":
        with open(fp, encoding="utf-8") as f:
            data = json.load(f)
        return pd.DataFrame(data) if isinstance(data, list) else data
    return None


datasets: dict[str, dict[str, pd.DataFrame]] = {}

for folder in sorted(DATA_DIR.iterdir()):
    if not folder.is_dir():
        continue
    print(f"\n=== {folder.name} ===")
    datasets[folder.name] = {}
    for fp in sorted(folder.rglob("*")):
        if not fp.is_file() or fp.suffix.lower() not in (".csv", ".json"):
            continue
        result = load_file(fp)
        rel = str(fp.relative_to(folder))
        if isinstance(result, pd.DataFrame):
            datasets[folder.name][rel] = result
            print(f"  {rel}: {result.shape[0]:,} rows × {result.shape[1]} cols")
        else:
            print(f"  {rel}: nested JSON — inspect manually")


=== inventory-data-for-pharmacy-website-in-json-format ===
  data.csv: 93 rows × 12 cols

=== pharma-sales-data ===
  salesdaily.csv: 2,106 rows × 13 cols
  saleshourly.csv: 50,532 rows × 13 cols
  salesmonthly.csv: 70 rows × 9 cols
  salesweekly.csv: 302 rows × 9 cols

=== pharmaceutical-supply-chain-optimization ===
  Pharmaceutical Supply Chain Optimization.csv: 100,000 rows × 4 cols

=== pharmacy-products-dataset ===
  Pharmacy_Products.csv: 6,198 rows × 5 cols


## Quick preview

In [12]:
for folder, files in datasets.items():
    for fname, df in files.items():
        print(f"\n--- {folder} / {fname} ---")
        display(df.head(3))


--- inventory-data-for-pharmacy-website-in-json-format / data.csv ---


,drugName,manufacturer,image,description,consumeType,expirydate,price,sideEffects,disclaimer,category,countInStock,expiryDate
0,Atorvastatin,Pfizer Inc.,https://cdn01.pharmeasy.in/dam/products/J21424...,Used to lower cholesterol and triglyceride lev...,Oral,2025-1-15,60,"Muscle pain, nausea, headache",Consult your doctor before use.,"HMG-CoA reductase inhibitors, or statins.",100,NaN
1,Amoxicillin,GlaxoSmithKline plc,https://5.imimg.com/data5/SELLER/Default/2023/...,Antibiotic used to treat bacterial infections.,Oral,2024-12-01,62,"Diarrhea, rash, nausea.",Complete the full course as prescribed.,antibiotics,100,NaN
2,Lisinopril,AstraZeneca plc,https://encrypted-tbn0.gstatic.com/images?q=tb...,Treats high blood pressure and heart failure.,Oral,2025-06-30,40,"Dizziness, headache, fatigue.",Monitor blood pressure regularly.,blood pressure,100,NaN



--- pharma-sales-data / salesdaily.csv ---


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday



--- pharma-sales-data / saleshourly.csv ---


...

,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,1/5/2014,14.00,11.67,21.3,185.95,41.0,0.0,32.0,7.0
1,1/12/2014,29.33,12.68,37.9,190.70,88.0,5.0,21.0,7.2
2,1/19/2014,30.67,26.34,45.9,218.40,80.0,8.0,29.0,12.0



--- pharmaceutical-supply-chain-optimization / Pharmaceutical Supply Chain Optimization.csv ---


,Drug,Demand_Forecast,Optimal_Stock_Level,Restocking_Strategy
0,Metformin,7750,4753,Monthly
1,Lisinopril,5136,9965,Quarterly
2,Metformin,3183,2933,Monthly



--- pharmacy-products-dataset / Pharmacy_Products.csv ---


,name,packaging,price,discounted_price,discount_percentage
0,Bronchicum S Elixir Dietary Supplement,100 ml,65.5,65.5,NaN
1,Maalox Antacid Oral Suspension Liquid Lemon Fl...,20 x 4.3 ml,114.0,114.0,NaN
2,Doliprane 1000mg Paracetamol for Pain Relief &...,15 tablets,45.6,45.6,NaN
